# xQTL–GWAS enrichment + colocalization

## Description

Replaces the legacy `SuSiE_enloc.ipynb` two-step `[xqtl_gwas_enrichment]` / `[susie_coloc]` flow with two thin SoS steps backed by the new pecotmr S4 API. Pairs an xQTL-side `QtlFineMappingResult` with a GWAS-side `GwasFineMappingResult` (or `GwasSumStats`) and runs the two pipelines back-to-back:

- `[enrichment]` &mdash; `qtlEnrichmentPipeline(gwasFineMappingResult, qtlFineMappingResult, ...)` produces a per-(gwasStudy, qtlContext) enrichment data.frame.
- `[coloc]` &mdash; `colocPipeline(qtlFineMappingResult, gwasInput, ..., enrichment = <step1 output>)` runs `coloc::coloc.bf_bf` per (QTL tuple, GWAS tuple, CS pair) with enrichment-scaled `p12` priors (the "enloc" variant). Pass `--skip-enrichment` to run plain coloc with the default p12.

Because the S4 FMR collections already encapsulate every (study, context, trait, method) tuple across regions, the workflow is two single-call steps &mdash; no per-gene fan-out, no Python overlap helpers, no manifest. Per-region fan-out (and the per-region coloc loop) lives inside `colocPipeline` itself.

## Inputs

- `--qtl-fine-mapping` &mdash; path to S4 `QtlFineMappingResult` RDS (output of `fine_mapping.ipynb` / `fineMappingPipeline`).
- `--gwas-fine-mapping` &mdash; path to S4 `GwasFineMappingResult` RDS (also a `fineMappingPipeline` output). `colocPipeline` will also accept a `GwasSumStats` RDS and inline-fine-map it.
- `--name` &mdash; identifier used in the output filenames.
- Optional enrichment knobs (`--num-gwas`, `--pi-qtl`, `--lambda`, `--imp-n`).
- Optional coloc knobs (`--p1`, `--p2`, `--p12`, `--p12-max`, `--filter-lbf-cs`, `--filter-lbf-cs-secondary`, `--no-adjust-pips`).
- `--skip-enrichment` &mdash; skip step 1 and run coloc with the default p12 (the non-enloc variant).

## Outputs

- `{cwd}/{name}.enrichment.rds` &mdash; data.frame, one row per (gwasStudy, qtlContext) pair (omitted when `--skip-enrichment`).
- `{cwd}/{name}.coloc.rds` &mdash; data.frame, one row per (QTL tuple, GWAS tuple, CS pair).


## Example

```bash
sos run pipeline/enloc.ipynb susie_enloc \
    --cwd output --modular-script-dir /path/to/code/script \
    --name protocol_example_enloc \
    --qtl-fine-mapping  output/fine_mapping/protocol_example.qtl_finemap.rds \
    --gwas-fine-mapping output/fine_mapping/protocol_example.gwas_finemap.rds
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: name = str
parameter: qtl_fine_mapping = path
parameter: gwas_fine_mapping = path
# --- step 1 (enrichment) --------------------------------------------
parameter: skip_enrichment = False
parameter: num_gwas = -1.0    # negative → NULL pass-through
parameter: pi_qtl   = -1.0    # negative → NULL pass-through
parameter: lambda_  = 1.0
parameter: imp_n    = 25
# --- step 2 (coloc) -------------------------------------------------
parameter: p1                       = 1e-4
parameter: p2                       = 1e-4
parameter: p12                      = 5e-6
parameter: p12_max                  = 1e-3
parameter: filter_lbf_cs            = False
parameter: filter_lbf_cs_secondary  = -1.0   # negative → NULL
parameter: no_adjust_pips           = False
parameter: finemapping_methods      = 'susie'
# --- infrastructure -------------------------------------------------
parameter: modular_script_dir = path('code/script')
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '30m'
parameter: mem = '16G'
parameter: numThreads = 1


In [ ]:
[susie_enloc_1 (enrichment)]
stop_if(skip_enrichment, '--skip-enrichment set; skipping qtlEnrichmentPipeline.')
input: qtl_fine_mapping, gwas_fine_mapping
output: f"{cwd}/{name}.enrichment.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/qtl_enrichment.R \
        --qtl-fine-mapping ${qtl_fine_mapping} \
        --gwas-fine-mapping ${gwas_fine_mapping} \
        ${'--num-gwas ' + str(num_gwas) if num_gwas >= 0 else ''} \
        ${'--pi-qtl ' + str(pi_qtl) if pi_qtl >= 0 else ''} \
        --lambda ${lambda_} \
        --imp-n ${imp_n} \
        --ncore ${numThreads} \
        --output ${_output}


In [ ]:
[susie_enloc_2 (coloc)]
enrich_path = f"{cwd}/{name}.enrichment.rds"
enrich_arg  = f"--enrichment {enrich_path}" if not skip_enrichment else ''
input: qtl_fine_mapping, gwas_fine_mapping
output: f"{cwd}/{name}.coloc.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/coloc.R \
        --qtl-fine-mapping ${qtl_fine_mapping} \
        --gwas-input ${gwas_fine_mapping} \
        ${enrich_arg} \
        ${'--filter-lbf-cs' if filter_lbf_cs else ''} \
        ${'--filter-lbf-cs-secondary ' + str(filter_lbf_cs_secondary) if filter_lbf_cs_secondary >= 0 else ''} \
        --p1 ${p1} \
        --p2 ${p2} \
        --p12 ${p12} \
        --p12-max ${p12_max} \
        ${'--no-adjust-pips' if no_adjust_pips else ''} \
        --finemapping-methods ${finemapping_methods} \
        --output ${_output}
